# ClipMind-AI Evidence Benchmarks
Run these cells to generate hard evidence (logs and outputs) that will remain saved in this notebook. Commit this notebook to GitHub to serve as proof of your system's performance metrics for your resume.

In [1]:
import os
import subprocess
import time

# You can place some test videos in this directory, or use the uploads directory
test_videos_dir = '../backend/uploads'
os.makedirs(test_videos_dir, exist_ok=True)

def extract_audio(video_path: str, output_path: str):
    subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-acodec", "libmp3lame", "-q:a", "2",
        "-ac", "1", "-ar", "16000",
        output_path,
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def human_mb(num_bytes: int) -> float:
    return round(num_bytes / (1024 * 1024), 2)

print("--- Audio Extraction Payload Benchmark ---")
video_exts = (".mp4", ".mov", ".mkv", ".avi", ".webm")
videos = sorted([os.path.join(test_videos_dir, f) for f in os.listdir(test_videos_dir) if f.lower().endswith(video_exts)])

if not videos:
    print(f"No videos found in {test_videos_dir}. Please add some and run again.")
else:
    results = []
    tmp_audio = "_bench_tmp_audio.mp3"
    
    for video_path in videos:
        video_size = os.path.getsize(video_path)
        t0 = time.time()
        try:
            extract_audio(video_path, tmp_audio)
            elapsed = time.time() - t0
            audio_size = os.path.getsize(tmp_audio)
            os.remove(tmp_audio)
            
            reduction_pct = round((1 - audio_size / video_size) * 100, 1)
            results.append(reduction_pct)
            print(f"{os.path.basename(video_path)}: {human_mb(video_size)}MB -> {human_mb(audio_size)}MB ({reduction_pct}% reduction in {round(elapsed, 2)}s)")
        except Exception as e:
            print(f"Failed {os.path.basename(video_path)}: {e}")
    
    if results:
        print(f"\nFINAL RESULT: Average Payload Reduction = {round(sum(results)/len(results), 1)}% across {len(results)} videos.")


--- Audio Extraction Payload Benchmark ---
1_youtube.mp4: 9.64MB -> 0.81MB (91.6% reduction in 2.61s)
2_youtube.mp4: 12.9MB -> 1.28MB (90.1% reduction in 0.52s)
3_youtube.mp4: 9.89MB -> 1.19MB (87.9% reduction in 0.48s)
4_CourseForge.mp4: 10.89MB -> 0.62MB (94.3% reduction in 0.22s)
5_youtube.mp4: 42.47MB -> 5.92MB (86.1% reduction in 1.94s)
6_youtube.mp4: 20.53MB -> 7.69MB (62.6% reduction in 1.9s)
7_youtube.mp4: 1.91MB -> 0.15MB (92.3% reduction in 0.09s)
8_youtube.mp4: 14.43MB -> 1.01MB (93.0% reduction in 0.42s)

FINAL RESULT: Average Payload Reduction = 87.2% across 8 videos.


In [2]:
import sys
import os
import time
from dotenv import load_dotenv

# Add backend to path so we can import the actual pipeline
sys.path.insert(0, os.path.abspath('../backend'))
load_dotenv(os.path.abspath('../backend/.env'))

try:
    from ai.pipeline import run_ai_pipeline
except ImportError as e:
    print("Could not import pipeline. Make sure you have installed the backend requirements.", e)
    run_ai_pipeline = None

print("--- AI Pipeline Reliability & Latency Benchmark ---")
if run_ai_pipeline and videos:
    successes = 0
    latencies = []
    
    for video_path in videos:
        name = os.path.basename(video_path)
        audio_tmp = f"_bench_{name}.mp3"
        print(f"\nProcessing {name}...")
        
        t0 = time.time()
        try:
            segments, full_text, summary, short_summary, key_moments, keywords = run_ai_pipeline(
                video_path, audio_tmp,
                generate_transcript=True,
                generate_summary=True,
                generate_key_moments=True,
            )
            elapsed = round(time.time() - t0, 2)
            ok = bool(full_text.strip()) and bool(summary.strip())
            
            if ok:
                successes += 1
                latencies.append(elapsed)
                print(f"  -> SUCCESS in {elapsed}s")
                print(f"  -> Transcript length: {len(full_text)} chars")
                print(f"  -> Key moments generated: {len(key_moments)}")
            else:
                print(f"  -> EMPTY_OUTPUT in {elapsed}s")
        except Exception as e:
            print(f"  -> FAILED: {e}")
        finally:
            if os.path.exists(audio_tmp):
                os.remove(audio_tmp)
    
    if latencies:
        success_rate = round(successes/len(videos)*100, 1)
        avg_latency = round(sum(latencies)/len(latencies), 2)
        print(f"\nFINAL RESULT: Reliability = {success_rate}% ({successes}/{len(videos)} succeeded)")
        print(f"FINAL RESULT: Average End-to-End Pipeline Latency = {avg_latency} seconds")
elif not videos:
    print("No videos found to run the pipeline benchmark.")


--- AI Pipeline Reliability & Latency Benchmark ---

Processing 1_youtube.mp4...
  -> SUCCESS in 5.27s
  -> Transcript length: 1125 chars
  -> Key moments generated: 6

Processing 2_youtube.mp4...
  -> SUCCESS in 4.26s
  -> Transcript length: 2891 chars
  -> Key moments generated: 5

Processing 3_youtube.mp4...
  -> SUCCESS in 4.57s
  -> Transcript length: 2765 chars
  -> Key moments generated: 6

Processing 4_CourseForge.mp4...
  -> SUCCESS in 4.17s
  -> Transcript length: 1395 chars
  -> Key moments generated: 5

Processing 5_youtube.mp4...
  -> SUCCESS in 41.4s
  -> Transcript length: 18531 chars
  -> Key moments generated: 7

Processing 6_youtube.mp4...
  -> SUCCESS in 44.12s
  -> Transcript length: 16107 chars
  -> Key moments generated: 9

Processing 7_youtube.mp4...
  -> SUCCESS in 14.28s
  -> Transcript length: 534 chars
  -> Key moments generated: 5

Processing 8_youtube.mp4...
  -> SUCCESS in 16.59s
  -> Transcript length: 1791 chars
  -> Key moments generated: 5

FINAL RESUL

In [5]:
import os
import time
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv(os.path.abspath('../backend/.env'))
client = Groq(api_key=os.getenv("GROQ_API_KEYS").split(",")[0].strip() if os.getenv("GROQ_API_KEYS") else None)

print("--- GPT-OSS-20b Token Generation Speed Benchmark ---")
if not client.api_key:
    print("API Key not found!")
else:
    prompt = "Explain the architecture of a modern AI-powered web application in extreme detail."
    
    print("Sending prompt to openai/gpt-oss-20b...")
    t0 = time.time()
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    
    elapsed = time.time() - t0
    
    completion_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens
    tokens_per_second = round(completion_tokens / elapsed, 2)
    
    print(f"\nTime Taken: {round(elapsed, 3)} seconds")
    print(f"Tokens Generated: {completion_tokens}")
    print(f"FINAL RESULT: Generation Speed = {tokens_per_second} Tokens/Second 🚀")


--- GPT-OSS-20b Token Generation Speed Benchmark ---
Sending prompt to openai/gpt-oss-20b...

Time Taken: 2.631 seconds
Tokens Generated: 2048
FINAL RESULT: Generation Speed = 778.54 Tokens/Second 🚀
